In [0]:
# Read accounts table and generate cards

from pyspark.sql.functions import col, concat, lit, when, rand

# Define paths
silver_accounts_path = "/Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta/silver/accounts"
silver_cards_path = "/Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta/silver/cards"

# Read accounts table
df_accounts = spark.read.format("delta").load(silver_accounts_path)

print(f"Total accounts: {df_accounts.count()}")
print("\nAccounts Schema:")
df_accounts.printSchema()
print("\nSample accounts:")
df_accounts.show(5, truncate=False)

Total accounts: 10

Accounts Schema:
root
 |-- account_id: string (nullable = true)
 |-- latest_balance: double (nullable = true)
 |-- account_type: string (nullable = true)
 |-- branch_id: string (nullable = true)


Sample accounts:
+-------------+----------------+------------+---------+
|account_id   |latest_balance  |account_type|branch_id|
+-------------+----------------+------------+---------+
|1196428'     |-1.68284755405E9|Savings     |B4       |
|1196711'     |-1.58691558925E9|Savings     |B1       |
|409000362497'|-1.90190209261E9|Savings     |B1       |
|409000405747'|-5.4826745865E8 |Savings     |B2       |
|409000425051'|-3.5673478865E8 |Savings     |B3       |
+-------------+----------------+------------+---------+
only showing top 5 rows


In [0]:
# Generate card_id and card_type columns

# Create card_id as "C_" + account_id
df_cards = df_accounts.withColumn(
    "card_id",
    concat(lit("C_"), col("account_id"))
)

# Randomly assign card_type (Debit or Credit)
df_cards = df_cards.withColumn(
    "card_type",
    when(rand() < 0.5, lit("Debit")).otherwise(lit("Credit"))
)

# Select only required columns
df_cards = df_cards.select("card_id", "account_id", "card_type")

print(f"Total cards generated: {df_cards.count()}")
print("\nCards with account relationship:")
df_cards.show(10, truncate=False)

Total cards generated: 10

Cards with account relationship:
+---------------+-------------+---------+
|card_id        |account_id   |card_type|
+---------------+-------------+---------+
|C_1196428'     |1196428'     |Debit    |
|C_1196711'     |1196711'     |Debit    |
|C_409000362497'|409000362497'|Debit    |
|C_409000405747'|409000405747'|Debit    |
|C_409000425051'|409000425051'|Credit   |
|C_409000438611'|409000438611'|Debit    |
|C_409000438620'|409000438620'|Debit    |
|C_409000493201'|409000493201'|Credit   |
|C_409000493210'|409000493210'|Credit   |
|C_409000611074'|409000611074'|Debit    |
+---------------+-------------+---------+



In [0]:
# Remove duplicates based on card_id

print(f"Before deduplication: {df_cards.count()}")
df_cards = df_cards.dropDuplicates(["card_id"])
print(f"After deduplication: {df_cards.count()}")

print("\nFinal Cards Schema:")
df_cards.printSchema()

print("\nCard Type Distribution:")
df_cards.groupBy("card_type").count().orderBy("card_type").show()

Before deduplication: 10
After deduplication: 10

Final Cards Schema:
root
 |-- card_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- card_type: string (nullable = false)


Card Type Distribution:
+---------+-----+
|card_type|count|
+---------+-----+
|   Credit|    3|
|    Debit|    7|
+---------+-----+



In [0]:
# Save as Delta table

df_cards.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(silver_cards_path)

print(f"✓ Cards table saved successfully to: {silver_cards_path}")

✓ Cards table saved successfully to: /Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta/silver/cards


In [0]:
# Verify the saved Delta table

df_verify = spark.read.format("delta").load(silver_cards_path)

print(f"Total cards in Delta table: {df_verify.count()}")
print("\nDelta Table Schema:")
df_verify.printSchema()

print("\nSample cards with foreign key relationship:")
df_verify.orderBy("card_id").show(10, truncate=False)

print("\nCard Type Distribution:")
df_verify.groupBy("card_type").count().orderBy("card_type").show()

Total cards in Delta table: 10

Delta Table Schema:
root
 |-- card_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- card_type: string (nullable = true)


Sample cards with foreign key relationship:
+---------------+-------------+---------+
|card_id        |account_id   |card_type|
+---------------+-------------+---------+
|C_1196428'     |1196428'     |Debit    |
|C_1196711'     |1196711'     |Debit    |
|C_409000362497'|409000362497'|Debit    |
|C_409000405747'|409000405747'|Debit    |
|C_409000425051'|409000425051'|Credit   |
|C_409000438611'|409000438611'|Debit    |
|C_409000438620'|409000438620'|Debit    |
|C_409000493201'|409000493201'|Credit   |
|C_409000493210'|409000493210'|Credit   |
|C_409000611074'|409000611074'|Debit    |
+---------------+-------------+---------+


Card Type Distribution:
+---------+-----+
|card_type|count|
+---------+-----+
|   Credit|    3|
|    Debit|    7|
+---------+-----+

